# Harmony4D trên Drive → Google Sheet + manifest JSON

Notebook duyệt thư mục `test` của Harmony4D trên Google Drive. Với mỗi camera, nó lấy File ID của GT (`poses3d.zip`), video gốc, tệp PKL và video WHAM. Kết quả được ghi vào một Google Sheet và một tệp JSON `{"test": [...]}`.

**Cách dùng**
1. Mở notebook trên Colab. Sửa ô **Cấu hình** nếu cần.
2. Chọn `Runtime → Run all`. Lần đầu Colab hỏi quyền truy cập tài khoản Google. Hãy chọn tài khoản có quyền **Editor** trên thư mục dataset, vì notebook cần upload `poses3d.zip` vào đó.
3. Sheet được mở (hoặc tạo) ngay đầu lượt chạy. Xong mỗi sequence, các dòng của nó được ghi ngay vào Sheet và tệp JSON trên Colab cũng được ghi lại. Nếu lượt chạy bị ngắt giữa chừng, phần đã xong vẫn còn.
4. Có thể chạy lại nhiều lần. `poses3d.zip` đã có thì được dùng lại, nên lần sau nhanh hơn nhiều. Sheet được ghi tiếp xuống dưới; cột `time` cho biết dòng thuộc lượt chạy nào. Tệp JSON được ghi đè bằng kết quả của lượt chạy mới nhất.
5. Khi Drive hoặc Sheets báo vượt giới hạn tốc độ (403 rate limit, 429) hay lỗi máy chủ (5xx), notebook tự chờ rồi thử lại.

**Cấu trúc thư mục mong đợi**
```
<gốc>/test/<nhóm>/<X>/processed_data/poses3d/*.npy
                     /<X>/exo/camY/harmony4D_X_camY_ffmpeg.mp4        (hoặc trong camY/images/)
                     /<X>/exo/camY/harmony4D_X_camY_wham/  ← tệp .pkl có X và Y trong tên
                                                           ← tệp output_*X_camY*.mp4
```
Ví dụ tên do WHAM tạo: `wham_harmony4D_041_grappling2_cam07_ffmpeg.pkl` và `output_harmony4D_041_grappling2_cam07_ffmpeg.mp4`.
- `processed_data` được tìm ở mọi độ sâu, tối đa 3 cấp dưới `test`. Vì vậy có hay không có thư mục `<nhóm>` (ví dụ `03_grappling2`) đều chạy được.
- Y giữ nguyên như trong tên thư mục: `cam07` cho Y = `07`.

**Về `skip`:** `skip = true` khi thiếu `pkl_file_ID` hoặc `z1`. Đây là nghĩa đang dùng trong repo: `true` = bỏ qua sequence này.

**Về `image_size`:** `[W, H]` là kích thước frame của **video gốc**, vì WHAM nhận chính video đó làm đầu vào. Repo dùng `image_size` để dựng lại camera của WHAM: tiêu cự `f = hypot(W, H)`, điểm chính ở tâm ảnh. Không lấy kích thước từ video WHAM, vì khi WHAM chạy có khung nhìn toàn cảnh, khung đó được ghép vào bên phải nên video WHAM rộng hơn ảnh gốc. Video WHAM chỉ dùng để đối chiếu.

## 1. Cấu hình
Sửa các hằng số dưới đây nếu cấu trúc trên Drive khác.

In [ ]:
ROOT_FOLDER_ID = "1UX3jW_pA2FoJRNsXsjwHBUk5QjmSAlTK"  # thư mục Harmony4D trên Drive
TEST_FOLDER_NAMES = ("test", "test_set")  # tên thư mục test (lấy cái tìm thấy đầu tiên)
MAX_DEPTH = 3              # processed_data nằm sâu tối đa bao nhiêu cấp dưới test
FPS = 20                   # giá trị "fps" ghi vào JSON
ZIP_NAME = "poses3d.zip"
SHEET_NAME = "harmony4d_test_index"          # Google Sheet trong My Drive (tạo nếu chưa có)
LOCAL_JSON = "/content/harmony4d_test.json"  # cũng được upload lên My Drive cùng tên
LOCAL_WORK = "/content/h4d_work"             # chỗ tạm để tải poses3d và nén
DOWNLOAD_WORKERS = 8       # số luồng tải song song
NUM_RETRIES = 6            # số lần thử lại khi Drive/Sheets báo lỗi tạm thời (chờ tối đa 64 s/lần)
TIMEZONE = "Asia/Ho_Chi_Minh"                # múi giờ cho cột time (Colab mặc định UTC)

Thư viện và hằng số nội bộ. Mọi thư viện đều có sẵn trên Colab. Cảnh báo `httplib2 transport does not support per-request timeout` vô hại nên được tắt cho log dễ đọc.

In [ ]:
import io, os, re, json, logging, random, shutil, subprocess, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from zoneinfo import ZoneInfo

import google.auth
import gspread
import pandas as pd
from google.auth.transport.requests import Request as AuthRequest
from google.colab import auth, files
from gspread.exceptions import APIError
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from tqdm.auto import tqdm

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"
GOOGLE_APPS_PREFIX = "application/vnd.google-apps."  # Docs, Sheets...: không tải được dạng tệp
CAM_RE = re.compile(r"^cam(\d+)$")
SHEET_HEADER = ["STT", "file PKL", "original_video", "wham_video", "GT", "cam", "action", "time",
                "image_size"]
VIDEO_FIELDS = "videoMediaMetadata(width,height,durationMillis)"  # siêu dữ liệu video của Drive
_tls = threading.local()  # mỗi luồng tải giữ một kết nối Drive riêng
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

## 2. Đăng nhập
`auth_services()` đăng nhập Google trong Colab rồi trả về ba đối tượng:
- `creds`: thông tin đăng nhập. Mỗi luồng tải dùng nó để tạo kết nối riêng.
- `drive`: Drive API v3, dùng để liệt kê, tải và upload tệp.
- `gc`: client `gspread`, dùng để ghi Google Sheets.

In [ ]:
def auth_services():
    """Đăng nhập Google trong Colab, trả về (creds, drive, gc)."""
    auth.authenticate_user()
    creds, _ = google.auth.default()
    drive = build("drive", "v3", credentials=creds, cache_discovery=False)
    gc = gspread.authorize(creds)
    return creds, drive, gc

## 3. Tiện ích Drive
`is_folder(item)` cho biết một phần tử Drive (dict có `mimeType`) có phải thư mục không.

In [ ]:
def is_folder(item):
    """True nếu phần tử Drive là thư mục."""
    return item.get("mimeType") == FOLDER_MIME

`list_children(drive, parent_id, name_contains=None)` liệt kê các phần tử con trực tiếp của một thư mục.
- Đọc hết mọi trang kết quả, có hỗ trợ Shared Drive.
- Shortcut được thay bằng tệp hoặc thư mục đích của nó, nên vẫn duyệt được như phần tử thật.
- Nếu có `name_contains`, Drive chỉ trả về phần tử có tên bắt đầu bằng chuỗi đó. Cách này tránh phải liệt kê hàng nghìn ảnh trong `images/`.
- Kết quả sắp theo tên; các tệp trùng tên thì tệp mới sửa gần nhất đứng trước.
- Với tệp video, kết quả kèm luôn siêu dữ liệu `videoMediaMetadata` (rộng, cao, thời lượng) của Drive, nên đọc kích thước video không tốn thêm lượt gọi nào.
- Mọi lệnh gọi Drive API đều tự thử lại tối đa `NUM_RETRIES` lần khi gặp 403 do vượt giới hạn tốc độ, 429 hoặc 5xx. Thư viện `googleapiclient` lo việc này qua tham số `num_retries`.

In [ ]:
def list_children(drive, parent_id, name_contains=None):
    """Mọi phần tử con trực tiếp của parent_id: [{id, name, mimeType}, ...]."""
    query = f"'{parent_id}' in parents and trashed = false"
    if name_contains:
        safe = name_contains.replace("\\", "\\\\").replace("'", "\\'")
        query += f" and name contains '{safe}'"
    fields = f"nextPageToken, files(id, name, mimeType, shortcutDetails, {VIDEO_FIELDS})"
    items, token = [], None
    while True:
        resp = drive.files().list(
            q=query, fields=fields, pageSize=1000, pageToken=token,
            orderBy="name,modifiedTime desc",
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute(num_retries=NUM_RETRIES)
        for f in resp.get("files", []):
            if f["mimeType"] == SHORTCUT_MIME:
                target = f.get("shortcutDetails", {})
                f = {"id": target.get("targetId"), "name": f["name"],
                     "mimeType": target.get("targetMimeType")}
            items.append(f)
        token = resp.get("nextPageToken")
        if not token:
            return items

`find_one(items, test)` trả về phần tử đầu tiên thoả điều kiện `test`, hoặc `None` nếu không có. Khi có nhiều phần tử khớp, nó in cảnh báo để bạn kiểm tra lại. Vì `list_children` đặt tệp mới nhất lên trước, gặp tệp trùng tên thì bản mới nhất được chọn.

In [ ]:
def find_one(items, test):
    """Phần tử đầu tiên thoả test(item), hoặc None."""
    matches = [f for f in items if test(f)]
    if len(matches) > 1:
        names = {f["name"] for f in matches}
        why = "trùng tên, lấy bản mới nhất" if len(names) == 1 else f"lấy '{matches[0]['name']}'"
        print(f"    ! {len(matches)} phần tử khớp ({', '.join(sorted(names))}): {why}")
    return matches[0] if matches else None

`download_file(creds, file_id, path)` tải một tệp Drive về `path`. Hàm này chạy song song nhiều luồng. Thư viện `googleapiclient` không an toàn khi nhiều luồng dùng chung một kết nối, nên mỗi luồng tự tạo kết nối Drive riêng.

In [ ]:
def download_file(creds, file_id, path):
    """Tải tệp file_id về path (mỗi luồng một kết nối Drive)."""
    if not hasattr(_tls, "drive"):
        _tls.drive = build("drive", "v3", credentials=creds, cache_discovery=False)
    request = _tls.drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with io.FileIO(path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=NUM_RETRIES)
    return path

`download_folder(drive, creds, folder_id, dst)` tải cả một thư mục Drive, kể cả thư mục con, về `dst`. Hàm làm theo hai bước:
1. Liệt kê hết các tệp.
2. Tải song song bằng `DOWNLOAD_WORKERS` luồng.

Tệp Google Docs/Sheets bị bỏ qua. Hàm trả về số tệp đã tải.

In [ ]:
def download_folder(drive, creds, folder_id, dst):
    """Tải đệ quy folder_id về dst, trả về số tệp đã tải."""
    jobs, stack = [], [(folder_id, dst)]
    while stack:
        fid, local = stack.pop()
        for f in list_children(drive, fid):
            path = os.path.join(local, f["name"])
            if is_folder(f):
                stack.append((f["id"], path))
            elif not f["mimeType"].startswith(GOOGLE_APPS_PREFIX):
                jobs.append((f["id"], path))
    os.makedirs(dst, exist_ok=True)
    with ThreadPoolExecutor(DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_file, creds, fid, path) for fid, path in jobs]
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc="  tải poses3d", leave=False):
            fut.result()
    return len(jobs)

`zip_and_upload(drive, src_dir, parent_id, zip_name)` nén `src_dir` rồi upload tệp zip vào thư mục Drive `parent_id`. Hàm trả về File ID của tệp zip.

Bên trong zip có sẵn thư mục gốc `poses3d/`, nên giải nén ra là được ngay `poses3d/*.npy`.

In [ ]:
def zip_and_upload(drive, src_dir, parent_id, zip_name):
    """Nén src_dir thành zip_name, upload vào parent_id, trả về file ID."""
    parent_dir, folder = os.path.split(src_dir.rstrip("/"))
    base = os.path.join(parent_dir, os.path.splitext(zip_name)[0])
    archive = shutil.make_archive(base, "zip", root_dir=parent_dir, base_dir=folder)
    media = MediaFileUpload(archive, mimetype="application/zip", resumable=True)
    created = drive.files().create(
        body={"name": zip_name, "parents": [parent_id]}, media_body=media,
        fields="id", supportsAllDrives=True,
    ).execute(num_retries=NUM_RETRIES)
    return created["id"]

### Thử lại khi bị giới hạn tốc độ
Google Sheets được ghi qua thư viện `gspread`. Thư viện này không tự thử lại như `googleapiclient`, nên notebook cần hai hàm riêng.

`is_transient(err)` cho biết một lỗi có phải lỗi tạm thời không. Lỗi tạm thời gồm: 429, 5xx, và 403 do vượt giới hạn tốc độ (*rate limit*). Lỗi 403 do thiếu quyền **không** phải lỗi tạm thời.

In [ ]:
def is_transient(err):
    """True nếu lỗi Drive (HttpError) / Sheets (APIError) đáng thử lại."""
    if isinstance(err, HttpError):
        status = int(err.resp.status)
    else:
        status = int(err.response.status_code)
    rate_limited = status == 403 and "rate limit" in str(err).lower()
    return status == 429 or status >= 500 or rate_limited

`with_retry(call, what)` gọi `call()`. Nếu gặp lỗi tạm thời, hàm chờ 2, 4, 8… giây (tối đa 64 giây) rồi thử lại, tối đa `NUM_RETRIES` lần. Lỗi khác, hoặc hết số lần thử, thì báo lỗi như bình thường. `what` là mô tả ngắn để in ra khi phải chờ.

In [ ]:
def with_retry(call, what):
    """call() và thử lại khi gặp lỗi tạm thời (chờ tăng dần)."""
    for attempt in range(NUM_RETRIES + 1):
        try:
            return call()
        except (HttpError, APIError) as e:
            if attempt == NUM_RETRIES or not is_transient(e):
                raise
            wait = min(2 ** (attempt + 1), 64) + random.random()
            print(f"  … {what}: lỗi tạm thời ({e}), thử lại sau {wait:.0f} s")
            time.sleep(wait)

## 4. GT: `poses3d.zip` (bước 2–3)
`ensure_poses3d_zip(drive, creds, processed_id, action)` trả về `z1`, tức File ID của `processed_data/poses3d.zip`.
- Nếu đã có `poses3d.zip` thì lấy ID của nó và dừng.
- Nếu chưa có thì tải `poses3d` về `LOCAL_WORK`, nén, rồi upload lên cùng cấp với `poses3d`.
- Nếu không có `poses3d`, `poses3d` rỗng, hoặc Drive báo lỗi (ví dụ không có quyền ghi) thì trả về `None`.

Thư mục tạm luôn bị xoá sau khi xong, để ổ đĩa Colab không bị đầy.

In [ ]:
def ensure_poses3d_zip(drive, creds, processed_id, action):
    """Bước 2–3: z1 = file ID của processed_data/poses3d.zip (None nếu không có)."""
    items = list_children(drive, processed_id)
    existing = find_one(items, lambda f: f["name"] == ZIP_NAME and not is_folder(f))
    if existing:
        print(f"  {ZIP_NAME} đã có: {existing['id']}")
        return existing["id"]
    poses = find_one(items, lambda f: f["name"] == "poses3d" and is_folder(f))
    if poses is None:
        print("  ! không có thư mục poses3d")
        return None
    work = os.path.join(LOCAL_WORK, action)
    shutil.rmtree(work, ignore_errors=True)
    try:
        n = download_folder(drive, creds, poses["id"], os.path.join(work, "poses3d"))
        if n == 0:
            print("  ! poses3d rỗng, không tạo zip")
            return None
        z1 = zip_and_upload(drive, os.path.join(work, "poses3d"), processed_id, ZIP_NAME)
        print(f"  đã tạo {ZIP_NAME} ({n} tệp): {z1}")
        return z1
    except HttpError as e:
        print(f"  ! lỗi Drive khi tạo {ZIP_NAME}: {e}")
        return None
    finally:
        shutil.rmtree(work, ignore_errors=True)

## 5. Tìm các sequence (bước 1)
`find_sequences(drive, test_id)` duyệt thư mục `test` theo từng tầng, sâu tối đa `MAX_DEPTH` cấp, để tìm các thư mục `processed_data`.
- Thư mục chứa `processed_data` chính là thư mục X. Tên của nó là `action`.
- Khi đã thấy `processed_data` trong X thì hàm không đi sâu thêm vào X. Nhờ vậy nó không duyệt `exo/` hay `images/`.

Hàm trả về danh sách `(X, id thư mục X, id processed_data)`, sắp theo X.

In [ ]:
def find_sequences(drive, test_id):
    """Bước 1: [(X, id thư mục X, id processed_data), ...] dưới test."""
    found, level = [], [("", test_id)]
    for _ in range(MAX_DEPTH):
        next_level = []
        for name, folder_id in level:
            children = [f for f in list_children(drive, folder_id) if is_folder(f)]
            processed = find_one(children, lambda f: f["name"] == "processed_data")
            if processed is not None:
                found.append((name, folder_id, processed["id"]))
            else:
                next_level += [(f["name"], f["id"]) for f in children]
        level = next_level
    return sorted(found)

## 6. Camera (bước 4–5)
`list_cam_folders(drive, exo_id)` lấy các thư mục `camY` trong `exo` và sắp theo số Y tăng dần.

Y được giữ dạng chuỗi như trong tên thư mục, ví dụ `"07"`. Như vậy tên tệp `harmony4D_X_cam07_...` khớp đúng.

In [ ]:
def list_cam_folders(drive, exo_id):
    """[(Y, thư mục camY), ...] trong exo, Y là chuỗi, sắp theo số."""
    cams = []
    for f in list_children(drive, exo_id):
        m = CAM_RE.match(f["name"])
        if m and is_folder(f):
            cams.append((m.group(1), f))
    return sorted(cams, key=lambda c: int(c[0]))

`find_original_video(drive, cam_items, action, cam)` tìm video gốc `harmony4D_X_camY_ffmpeg*.mp4` và trả về phần tử Drive của nó (dict có `id`, tức giá trị `h4d_X_camY`, kèm siêu dữ liệu video), hoặc `None`.
1. Tìm trong `camY` trước.
2. Nếu không thấy, tìm tiếp trong `camY/images/`, vì notebook `render_mp4_colab.ipynb` mặc định ghi video vào đó. Bước này lọc theo tên ngay trên Drive, nên không phải liệt kê ảnh. Nếu có nhiều thư mục `images` trùng tên, hàm tìm lần lượt trong từng thư mục.

In [ ]:
def find_original_video(drive, cam_items, action, cam):
    """Bước 4a: phần tử Drive của harmony4D_X_camY_ffmpeg*.mp4, hoặc None."""
    stem = f"harmony4D_{action}_cam{cam}_ffmpeg"
    test = lambda f: f["name"].startswith(stem) and f["name"].lower().endswith(".mp4")
    video = find_one(cam_items, test)
    image_dirs = [f for f in cam_items if f["name"] == "images" and is_folder(f)]
    for images in image_dirs:
        if video is not None:
            break
        video = find_one(list_children(drive, images["id"], stem), test)
    return video

`find_wham_files(drive, cam_items, action, cam)` vào thư mục `harmony4D_X_camY_wham` trong `camY` và tìm hai tệp:
- tệp `.pkl` có cả X và Y trong tên → `pkl_file_ID`
- tệp `.mp4` có tên bắt đầu bằng `output_` và chứa `X_camY` → `output_wham_video`. Tên thật do WHAM tạo là `output_harmony4D_X_camY_ffmpeg.mp4`: chữ `harmony4D_` nằm giữa `output_` và X, nên không thể đòi chuỗi `output_X_camY` đứng liền nhau.

Hàm trả về `(pkl_file_ID, video_WHAM, ghi_chú)`. `video_WHAM` là phần tử Drive (dict có `id`), để còn đối chiếu kích thước với video gốc. Tệp nào không có thì ứng với `None`. Ghi chú giải thích vì sao thiếu: không có thư mục `_wham`, hoặc thư mục `_wham` đang chứa những tệp nào.

In [ ]:
def find_wham_files(drive, cam_items, action, cam):
    """Bước 4b + 5: (pkl_file_ID, phần tử video WHAM, ghi chú), thiếu thì None."""
    wham_name = f"harmony4D_{action}_cam{cam}_wham"
    wham = find_one(cam_items, lambda f: f["name"].lower() == wham_name.lower()
                    and is_folder(f))
    if wham is None:
        return None, None, "chưa có thư mục _wham"
    items = [f for f in list_children(drive, wham["id"]) if not is_folder(f)]
    pkl = find_one(items, lambda f: f["name"].lower().endswith(".pkl")
                   and action in f["name"] and cam in f["name"])
    video = find_one(items, lambda f: f["name"].lower().endswith(".mp4")
                     and f["name"].startswith("output_") and f"{action}_cam{cam}" in f["name"])
    note = ""
    if pkl is None or video is None:
        names = [f["name"] for f in items]
        more = f" (+{len(names) - 4})" if len(names) > 4 else ""
        note = "_wham có: " + (", ".join(names[:4]) + more if names else "(rỗng)")
    return (pkl or {}).get("id"), video, note

### Kích thước video
`video_info(drive, creds, item)` trả về `(W, H, thời lượng ms)` của một video trên Drive. Kích thước video không đổi theo frame, nên chỉ cần đọc thông tin ở đầu tệp, không cần giải mã frame nào.
1. Dùng siêu dữ liệu `videoMediaMetadata` có sẵn trong kết quả liệt kê thư mục.
2. Nếu kết quả liệt kê không có (ví dụ phần tử là shortcut), hỏi riêng siêu dữ liệu của tệp đó, một lượt gọi.
3. Nếu Drive vẫn chưa có (video chưa được Drive xử lý xong), đọc bằng `ffprobe_video`.

In [ ]:
def video_info(drive, creds, item):
    """(W, H, thời lượng ms) của video Drive, hoặc None nếu không đọc được."""
    meta = item.get("videoMediaMetadata")
    if meta is None:
        meta = drive.files().get(
            fileId=item["id"], fields=VIDEO_FIELDS, supportsAllDrives=True,
        ).execute(num_retries=NUM_RETRIES).get("videoMediaMetadata")
    if meta and meta.get("width") and meta.get("height"):
        return int(meta["width"]), int(meta["height"]), int(meta.get("durationMillis") or 0)
    return ffprobe_video(creds, item["id"])

`ffprobe_video(creds, file_id)` chạy `ffprobe` (có sẵn trên Colab) thẳng trên link tải của Drive, kèm token đăng nhập. `ffprobe` chỉ tải phần header của mp4, thường vài trăm KB, không tải cả video.

Hàm trả về `(W, H, thời lượng ms)`, hoặc `None` nếu không đọc được. Khi báo lỗi, hàm chỉ in thông báo của `ffprobe`, không in câu lệnh, để token không lộ ra log.

In [ ]:
def ffprobe_video(creds, file_id):
    """(W, H, thời lượng ms) đọc bằng ffprobe trên link Drive, hoặc None."""
    if not creds.valid:
        creds.refresh(AuthRequest())
    url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media&supportsAllDrives=true"
    cmd = ["ffprobe", "-v", "error", "-headers", f"Authorization: Bearer {creds.token}\r\n",
           "-select_streams", "v:0", "-show_entries", "stream=width,height:format=duration",
           "-of", "json", url]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    except (OSError, subprocess.TimeoutExpired):
        print(f"    ! ffprobe không chạy được hoặc quá 120 s ({file_id})")
        return None
    try:
        info = json.loads(r.stdout)
        s = info["streams"][0]
        duration = float(info.get("format", {}).get("duration") or 0)
        return int(s["width"]), int(s["height"]), int(duration * 1000)
    except (ValueError, KeyError, IndexError):
        print(f"    ! ffprobe không đọc được video {file_id}: {r.stderr.strip()[-200:]}")
        return None

`check_wham_video(original, wham)` so video WHAM với video gốc, mỗi bên là `(W, H, thời lượng ms)`. Nếu khớp, hàm trả về chuỗi rỗng. Nếu lệch, hàm trả về cảnh báo, gợi ý video WHAM có thể không thuộc camera này.
- Chiều cao phải bằng nhau.
- Chiều rộng của video WHAM không được nhỏ hơn video gốc. Rộng hơn là bình thường khi có ghép khung nhìn toàn cảnh.
- Thời lượng lệch nhau không quá 1 giây (hoặc 2% nếu video dài).

In [ ]:
def check_wham_video(original, wham):
    """Cảnh báo nếu video WHAM lệch video gốc, chuỗi rỗng nếu khớp."""
    (W, H, dur), (w2, h2, dur2) = original, wham
    problems = []
    if h2 != H:
        problems.append(f"cao {h2} ≠ {H}")
    if w2 < W:
        problems.append(f"rộng {w2} < {W}")
    if dur and dur2 and abs(dur - dur2) > max(1000, 0.02 * dur):
        problems.append(f"dài {dur2 / 1000:.1f} s ≠ {dur / 1000:.1f} s")
    return ("video WHAM lệch video gốc: " + ", ".join(problems)) if problems else ""

`process_cam(drive, creds, action, cam, cam_folder, z1)` xử lý một thư mục `camY` và trả về một dòng kết quả dạng dict.
- `image_size` lấy từ video gốc. Không có video gốc, hoặc không đọc được kích thước, thì để `None`.
- Nếu có cả video gốc và video WHAM thì đối chiếu hai video bằng `check_wham_video`.

Hàm in một dòng tóm tắt: ✓ là tìm thấy, ✗ là thiếu, kèm kích thước video gốc. Nếu thiếu tệp hoặc video WHAM lệch thì in thêm ghi chú.

In [ ]:
def process_cam(drive, creds, action, cam, cam_folder, z1):
    """Một thư mục camY → một dòng kết quả (dict)."""
    items = list_children(drive, cam_folder["id"])
    original = find_original_video(drive, items, action, cam)
    pkl_id, wham_video, note = find_wham_files(drive, items, action, cam)
    info = video_info(drive, creds, original) if original else None
    if info and wham_video:
        wham_info = video_info(drive, creds, wham_video)
        warn = check_wham_video(info, wham_info) if wham_info else "không đọc được video WHAM"
        note = "; ".join(x for x in (note, warn) if x)
    size = f" {info[0]}x{info[1]}" if info else (" ?x?" if original else "")
    mark = lambda v: "✓" if v else "✗"
    print(f"    cam{cam}: pkl {mark(pkl_id)}  video gốc {mark(original)}{size}  "
          f"video WHAM {mark(wham_video)}" + (f"  ({note})" if note else ""))
    return {"action": action, "cam": cam, "pkl_file_ID": pkl_id,
            "original_video": (original or {}).get("id"),
            "output_wham_video": (wham_video or {}).get("id"), "z1": z1,
            "image_size": [info[0], info[1]] if info else None}

## 7. Một sequence
`process_action(drive, creds, action, action_id, processed_id)` xử lý một sequence X:
1. Lấy `z1` (bước 2–3).
2. Vào `exo` và xử lý từng `camY` (bước 4–5).

Hàm trả về danh sách dòng, mỗi camera một dòng.

In [ ]:
def process_action(drive, creds, action, action_id, processed_id):
    """Một sequence X → danh sách dòng kết quả (mỗi camera một dòng)."""
    print(f"\n[{action}]")
    z1 = ensure_poses3d_zip(drive, creds, processed_id, action)
    exo = find_one(list_children(drive, action_id),
                   lambda f: f["name"] == "exo" and is_folder(f))
    if exo is None:
        print("  ! không có thư mục exo")
        return []
    return [process_cam(drive, creds, action, cam, folder, z1)
            for cam, folder in list_cam_folders(drive, exo["id"])]

## 8. Xuất kết quả (bước 6–7)
`make_json_entry(row)` tạo một phần tử của danh sách `"test"`.
- ID nào thiếu thì ghi chuỗi rỗng `""`.
- `skip = true` khi thiếu `pkl_file_ID` hoặc `z1`.
- `image_size = [W, H]` của video gốc; không xác định được thì `null`. Repo coi `null` như thiếu khoá và chỉ báo lỗi khi thật sự cần chiếu WHAM lên ảnh.

In [ ]:
def make_json_entry(row):
    """Bước 7: một phần tử của danh sách "test"."""
    return {
        "name": f"h4d_{row['action']}_cam{row['cam']}",
        "wham_pkl": row["pkl_file_ID"] or "",
        "gt": {"type": "harmony4d", "path": row["z1"] or ""},
        "output_wham_video": row["output_wham_video"] or "",
        "skip": not (row["pkl_file_ID"] and row["z1"]),
        "fps": FPS,
        "image_size": row["image_size"],
    }

`open_sheet(gc)` mở Google Sheet `SHEET_NAME` trong My Drive, chưa có thì tạo mới. Sheet trống thì hàm ghi dòng tiêu đề: STT, file PKL, original_video, wham_video, GT, cam, action, time, image_size.

Sheet tạo từ bản notebook cũ có dòng tiêu đề thiếu cột `image_size`. Khi đó hàm ghi lại dòng tiêu đề cho đủ cột. Nếu dòng tiêu đề khác hẳn thì hàm chỉ in cảnh báo, không sửa.

Hàm được gọi **ngay đầu lượt chạy**, nên nếu không mở được Sheet thì bạn biết ngay, không phải chờ hết cả lượt. Mọi lệnh gọi Sheets đều đi qua `with_retry`. Hàm trả về trang tính đầu tiên.

In [ ]:
def open_sheet(gc):
    """Mở (hoặc tạo) SHEET_NAME, ghi dòng tiêu đề nếu trống; trả về sheet1."""
    try:
        sh = with_retry(lambda: gc.open(SHEET_NAME), "mở Sheet")
    except gspread.SpreadsheetNotFound:
        sh = with_retry(lambda: gc.create(SHEET_NAME), "tạo Sheet")
    ws = sh.sheet1
    header = with_retry(lambda: ws.row_values(1), "đọc Sheet")
    if not header:
        with_retry(lambda: ws.append_row(SHEET_HEADER, value_input_option="RAW"),
                   "ghi tiêu đề Sheet")
    elif header == SHEET_HEADER[:len(header)] and len(header) < len(SHEET_HEADER):
        with_retry(lambda: ws.update(range_name="A1", values=[SHEET_HEADER],
                                     value_input_option="RAW"), "cập nhật tiêu đề Sheet")
    elif header != SHEET_HEADER:
        print(f"! Dòng tiêu đề của Sheet khác {SHEET_HEADER}, hãy kiểm tra lại")
    print(f"Sheet: https://docs.google.com/spreadsheets/d/{sh.id}")
    return ws

`append_sheet_rows(ws, rows, now)` ghi các dòng của **một** sequence xuống cuối Sheet (bước 6). Hàm được gọi ngay sau khi xử lý xong mỗi sequence.
- STT đánh tiếp từ số dòng đã có.
- Giá trị được ghi nguyên văn (RAW), nên ID không bị đổi thành số và `07` không mất số 0.

In [ ]:
def append_sheet_rows(ws, rows, now):
    """Bước 6: ghi tiếp rows xuống cuối Sheet, STT đánh tiếp."""
    if not rows:
        return
    start = len(with_retry(lambda: ws.col_values(1), "đọc Sheet"))  # kể cả tiêu đề
    size = lambda r: "x".join(map(str, r["image_size"])) if r["image_size"] else ""
    values = [[start + i, r["pkl_file_ID"] or "", r["original_video"] or "",
               r["output_wham_video"] or "", r["z1"] or "", r["cam"], r["action"], now,
               size(r)] for i, r in enumerate(rows)]
    with_retry(lambda: ws.append_rows(values, value_input_option="RAW"), "ghi Sheet")

`save_json_local(entries, local_path)` ghi nội dung `{"test": entries}` vào `local_path` trên Colab. Hàm được gọi sau mỗi sequence, nên lượt chạy bị ngắt thì tệp vẫn giữ phần đã xong.

In [ ]:
def save_json_local(entries, local_path):
    """Ghi {"test": entries} ra local_path (ghi đè)."""
    with open(local_path, "w", encoding="utf-8") as fh:
        json.dump({"test": entries}, fh, ensure_ascii=False, indent=2)

`upload_json(drive, local_path)` upload tệp JSON lên thư mục gốc My Drive. Nếu đã có tệp cùng tên thì ghi đè. Hàm trả về File ID trên Drive.

In [ ]:
def upload_json(drive, local_path):
    """Upload local_path lên My Drive (ghi đè tệp cùng tên), trả về file ID."""
    name = os.path.basename(local_path)
    media = MediaFileUpload(local_path, mimetype="application/json")
    old = drive.files().list(
        q=f"name = '{name}' and 'root' in parents and trashed = false",
        fields="files(id)",
    ).execute(num_retries=NUM_RETRIES).get("files", [])
    if old:
        req = drive.files().update(fileId=old[0]["id"], media_body=media, fields="id")
    else:
        req = drive.files().create(body={"name": name}, media_body=media, fields="id")
    return req.execute(num_retries=NUM_RETRIES)["id"]

## 9. Chạy
`main()` gọi lần lượt bước 1 đến bước 7:
1. Mở Sheet trước, rồi tìm các sequence.
2. Với mỗi sequence: xử lý xong thì ghi ngay các dòng của nó vào Sheet và ghi lại tệp JSON trên Colab.
3. Cuối cùng upload tệp JSON lên My Drive.

Nếu Drive báo lỗi ở một sequence (sau khi đã thử lại), sequence đó bị bỏ qua và các sequence còn lại vẫn chạy. Cả lượt chạy dùng chung một mốc `time`. Hàm trả về `(rows, entries)`.

In [ ]:
def main():
    """Chạy bước 1–7, ghi Sheet và JSON sau mỗi sequence; trả về (rows, entries)."""
    creds, drive, gc = auth_services()
    ws = open_sheet(gc)
    now = datetime.now(ZoneInfo(TIMEZONE)).strftime("%Y-%m-%d %H:%M:%S")
    test = find_one(list_children(drive, ROOT_FOLDER_ID),
                    lambda f: f["name"] in TEST_FOLDER_NAMES and is_folder(f))
    if test is None:
        raise SystemExit(f"Không thấy thư mục {TEST_FOLDER_NAMES} trong {ROOT_FOLDER_ID}")
    sequences = find_sequences(drive, test["id"])
    print(f"Tìm thấy {len(sequences)} sequence có processed_data trong '{test['name']}'")
    rows, entries = [], []
    for action, action_id, processed_id in sequences:
        try:
            new = process_action(drive, creds, action, action_id, processed_id)
        except HttpError as e:
            print(f"  ! bỏ qua {action}: {e}")
            continue
        rows += new
        entries += [make_json_entry(r) for r in new]
        append_sheet_rows(ws, new, now)
        save_json_local(entries, LOCAL_JSON)
    save_json_local(entries, LOCAL_JSON)
    file_id = upload_json(drive, LOCAL_JSON)
    n_ok = sum(not e["skip"] for e in entries)
    print(f"\nXong: {len(rows)} camera, {n_ok} dùng được (skip = false).")
    print(f"JSON : {LOCAL_JSON} (Drive ID {file_id})")
    return rows, entries

Chạy toàn bộ quy trình, tải tệp JSON về máy và hiện bảng kết quả.

Nếu lượt chạy bị ngắt giữa chừng, Sheet và tệp `LOCAL_JSON` trên Colab vẫn giữ các sequence đã xong. Có thể tải tệp đó về bằng `files.download(LOCAL_JSON)`.

In [ ]:
rows, entries = main()
files.download(LOCAL_JSON)
pd.DataFrame(rows)